# Çeviri Sonuçları Görselleştirme

Bu notebook deney sonuçlarını görselleştirir ve karşılaştırır.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

In [ ]:
# En son deney sonucunu yükle
results_dir = Path('../../data/results')
result_files = sorted(results_dir.glob('exp_*.json'))

if not result_files:
    print("Henüz deney sonucu yok!")
else:
    latest_result = result_files[-1]
    print(f"Yükleniyor: {latest_result.name}")
    
    with open(latest_result, 'r', encoding='utf-8') as f:
        experiment = json.load(f)
    
    print(f"\nDeney ID: {experiment['experiment_id']}")
    print(f"Tarih: {experiment['timestamp']}")
    print(f"Toplam segment: {len(experiment['results'])}")
    print(f"Araçlar: {', '.join(experiment['config']['translators'])}")

In [ ]:
# Sonuçları DataFrame'e çevir
results_data = []

for result in experiment['results']:
    for tool, metrics in result.get('metrics', {}).items():
        if metrics:
            results_data.append({
                'segment_id': result['segment_id'],
                'category': result['category'],
                'length': result['length'],
                'tool': tool,
                'bleu': metrics.get('bleu'),
                'meteor': metrics.get('meteor'),
                'ter': metrics.get('ter'),
                'chrf': metrics.get('chrf')
            })

df = pd.DataFrame(results_data)
print(f"Toplam çeviri: {len(df)}")
df.head()

## Araç Karşılaştırması

In [ ]:
# Ortalama skorlar
avg_scores = df.groupby('tool')[['bleu', 'meteor', 'ter', 'chrf']].mean()
print("Ortalama Skorlar:")
print(avg_scores)

# Bar chart
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = ['bleu', 'meteor', 'ter', 'chrf']
titles = ['BLEU Skorları', 'METEOR Skorları', 'TER Skorları (Düşük=İyi)', 'chrF++ Skorları']

for idx, (metric, title) in enumerate(zip(metrics, titles)):
    ax = axes[idx // 2, idx % 2]
    avg_scores[metric].plot(kind='bar', ax=ax, color=['#3b82f6', '#6366f1', '#06b6d4', '#f97316'])
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Araç')
    ax.set_ylabel('Skor')
    ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../../paper/figures/tool_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Kategori Bazlı Analiz

In [ ]:
# Kategoriye göre BLEU skorları
category_scores = df.pivot_table(values='bleu', index='category', columns='tool', aggfunc='mean')
print("\nKategoriye Göre BLEU Skorları:")
print(category_scores)

category_scores.plot(kind='bar', figsize=(12, 6))
plt.title('Kategoriye Göre BLEU Skorları', fontsize=14, fontweight='bold')
plt.xlabel('Kategori')
plt.ylabel('BLEU Skoru')
plt.legend(title='Araç')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../../paper/figures/category_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Box Plot Analizi

In [ ]:
# BLEU skorlarının dağılımı
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='tool', y='bleu', palette='Set2')
plt.title('BLEU Skor Dağılımı (Box Plot)', fontsize=14, fontweight='bold')
plt.xlabel('Araç')
plt.ylabel('BLEU Skoru')
plt.tight_layout()
plt.savefig('../../paper/figures/bleu_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## Korelasyon Analizi

In [ ]:
# Metrikler arası korelasyon
correlation = df[['bleu', 'meteor', 'ter', 'chrf']].corr()
print("\nMetrikler Arası Korelasyon:")
print(correlation)

plt.figure(figsize=(10, 8))
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Metrikler Arası Korelasyon Matrisi', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../../paper/figures/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

## Uzunluk vs Kalite

In [ ]:
# Metin uzunluğu vs BLEU skoru
plt.figure(figsize=(12, 6))

for tool in df['tool'].unique():
    tool_data = df[df['tool'] == tool]
    plt.scatter(tool_data['length'], tool_data['bleu'], alpha=0.5, label=tool, s=20)

plt.title('Metin Uzunluğu vs BLEU Skoru', fontsize=14, fontweight='bold')
plt.xlabel('Metin Uzunluğu (karakter)')
plt.ylabel('BLEU Skoru')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../../paper/figures/length_vs_quality.png', dpi=300, bbox_inches='tight')
plt.show()

## Özet Tablo

In [ ]:
# Kapsamlı özet tablo
summary_table = df.groupby('tool').agg({
    'bleu': ['mean', 'std', 'min', 'max'],
    'meteor': ['mean', 'std'],
    'ter': ['mean', 'std'],
    'chrf': ['mean', 'std']
})

print("\nKapsamlı Özet Tablo:")
print(summary_table)

# LaTeX formatında export
print("\n\nLaTeX Formatı:")
print(summary_table.to_latex(float_format="%.4f"))